In [1]:
import compresso, torch
import numpy as np
import pandas as pd

from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Iterator, Mapping, Optional, Sequence, Union

import itertools
import multiprocessing.dummy
import queue
import random
import threading
import warnings
import weakref
from contextlib import closing

import numpy as np


ArrayLike = Union[np.ndarray, Sequence[int], Sequence[str]]

# taken from keras https://github.com/keras-team/keras/blob/v3.3.3/keras/src/trainers/data_adapters/py_dataset_adapter.py#L18-L175

class PyDataset:
    """Base class for defining a parallel dataset using Python code.

    Every `PyDataset` must implement the `__getitem__()` and the `__len__()`
    methods. If you want to modify your dataset between epochs,
    you may additionally implement `on_epoch_end()`,
    or `on_epoch_begin` to be called at the start of each epoch.
    The `__getitem__()` method should return a complete batch
    (not a single sample), and the `__len__` method should return
    the number of batches in the dataset (rather than the number of samples).

    Args:
        workers: Number of workers to use in multithreading or
            multiprocessing.
        use_multiprocessing: Whether to use Python multiprocessing for
            parallelism. Setting this to `True` means that your
            dataset will be replicated in multiple forked processes.
            This is necessary to gain compute-level (rather than I/O level)
            benefits from parallelism. However it can only be set to
            `True` if your dataset can be safely pickled.
        max_queue_size: Maximum number of batches to keep in the queue
            when iterating over the dataset in a multithreaded or
            multiprocessed setting.
            Reduce this value to reduce the CPU memory consumption of
            your dataset. Defaults to 10.

    Notes:

    - `PyDataset` is a safer way to do multiprocessing.
        This structure guarantees that the model will only train
        once on each sample per epoch, which is not the case
        with Python generators.
    - The arguments `workers`, `use_multiprocessing`, and `max_queue_size`
        exist to configure how `fit()` uses parallelism to iterate
        over the dataset. They are not being used by the `PyDataset` class
        directly. When you are manually iterating over a `PyDataset`,
        no parallelism is applied.

    Example:

    ```python
    from skimage.io import imread
    from skimage.transform import resize
    import numpy as np
    import math

    # Here, `x_set` is list of path to the images
    # and `y_set` are the associated classes.

    class CIFAR10PyDataset(keras.utils.PyDataset):

        def __init__(self, x_set, y_set, batch_size, **kwargs):
            super().__init__(**kwargs)
            self.x, self.y = x_set, y_set
            self.batch_size = batch_size

        def __len__(self):
            # Return number of batches.
            return math.ceil(len(self.x) / self.batch_size)

        def __getitem__(self, idx):
            # Return x, y for batch idx.
            low = idx * self.batch_size
            # Cap upper bound at array length; the last batch may be smaller
            # if the total number of items is not a multiple of batch size.
            high = min(low + self.batch_size, len(self.x))
            batch_x = self.x[low:high]
            batch_y = self.y[low:high]

            return np.array([
                resize(imread(file_name), (200, 200))
                   for file_name in batch_x]), np.array(batch_y)
    ```
    """

    def __init__(self, workers=1, use_multiprocessing=False, max_queue_size=10):
        self._workers = workers
        self._use_multiprocessing = use_multiprocessing
        self._max_queue_size = max_queue_size

    def _warn_if_super_not_called(self):
        warn = False
        if not hasattr(self, "_workers"):
            self._workers = 1
            warn = True
        if not hasattr(self, "_use_multiprocessing"):
            self._use_multiprocessing = False
            warn = True
        if not hasattr(self, "_max_queue_size"):
            self._max_queue_size = 10
            warn = True
        if warn:
            warnings.warn(
                "Your `PyDataset` class should call "
                "`super().__init__(**kwargs)` in its constructor. "
                "`**kwargs` can include `workers`, "
                "`use_multiprocessing`, `max_queue_size`. Do not pass "
                "these arguments to `fit()`, as they will be ignored.",
                stacklevel=2,
            )

    @property
    def workers(self):
        self._warn_if_super_not_called()
        return self._workers

    @workers.setter
    def workers(self, value):
        self._workers = value

    @property
    def use_multiprocessing(self):
        self._warn_if_super_not_called()
        return self._use_multiprocessing

    @use_multiprocessing.setter
    def use_multiprocessing(self, value):
        self._use_multiprocessing = value

    @property
    def max_queue_size(self):
        self._warn_if_super_not_called()
        return self._max_queue_size

    @max_queue_size.setter
    def max_queue_size(self, value):
        self._max_queue_size = value

    def __getitem__(self, index):
        """Gets batch at position `index`.

        Args:
            index: position of the batch in the PyDataset.

        Returns:
            A batch
        """
        del index
        raise NotImplementedError

    def __iter__(self):
        index_range = None
        try:
            num_batches = self.num_batches
            if num_batches is not None:
                index_range = range(num_batches)
        except NotImplementedError:
            pass

        if index_range is None:
            index_range = itertools.count()

        for index in index_range:
            yield self[index]

    @property
    def num_batches(self):
        """Number of batches in the PyDataset.

        Returns:
            The number of batches in the PyDataset or `None` to indicate that
            the dataset is infinite.
        """
        # For backwards compatibility, support `__len__`.
        if hasattr(self, "__len__"):
            return len(self)
        raise NotImplementedError(
            "You need to implement the `num_batches` property:\n\n"
            "@property\ndef num_batches(self):\n  return ..."
        )

    def on_epoch_begin(self):
        """Method called at the beginning of every epoch."""
        pass

    def on_epoch_end(self):
        """Method called at the end of every epoch."""
        pass


In [2]:
help(compresso.nn.sae.TopKSAE)

Help on class TopKSAE in module compresso.nn.sae:

class TopKSAE(torch.nn.modules.module.Module)
 |  TopKSAE(input_dim: 'int', hidden_dim: 'int', k: 'int', tied: 'bool' = False, pre_act: 'Optional[nn.Module]' = None) -> 'None'
 |  
 |  Sparse autoencoder with hard top-k bottleneck.
 |  
 |  Parameters
 |  ----------
 |  input_dim : int
 |      Dimensionality of the input (and reconstruction).
 |  hidden_dim : int
 |      Width of the sparse code layer.
 |  k : int
 |      Number of active features per sample.
 |  tied : bool
 |      If ``True``, decoder weight is the transpose of the encoder weight.
 |  pre_act : nn.Module | None
 |      Optional activation applied to encoder output *before* sparsification.
 |  
 |  Method resolution order:
 |      TopKSAE
 |      torch.nn.modules.module.Module
 |      builtins.object
 |  
 |  Methods defined here:
 |  
 |  __init__(self, input_dim: 'int', hidden_dim: 'int', k: 'int', tied: 'bool' = False, pre_act: 'Optional[nn.Module]' = None) -> 'Non

In [3]:
import torchvision

In [4]:
mnist=torchvision.datasets.MNIST("mnist-data", download=True)

In [5]:
datapoint=mnist[0]

im, label = datapoint
print(label)
im

5


In [6]:
mnist.data.shape,mnist.targets.shape

(torch.Size([60000, 28, 28]), torch.Size([60000]))

In [7]:
batch=mnist.data[:128]

In [30]:
torch.flatten(batch, start_dim=1)

torch.Size([128, 784])

In [34]:
help(PyDataset)

Help on class PyDataset in module __main__:

class PyDataset(builtins.object)
 |  PyDataset(workers=1, use_multiprocessing=False, max_queue_size=10)
 |  
 |  Base class for defining a parallel dataset using Python code.
 |  
 |  Every `PyDataset` must implement the `__getitem__()` and the `__len__()`
 |  methods. If you want to modify your dataset between epochs,
 |  you may additionally implement `on_epoch_end()`,
 |  or `on_epoch_begin` to be called at the start of each epoch.
 |  The `__getitem__()` method should return a complete batch
 |  (not a single sample), and the `__len__` method should return
 |  the number of batches in the dataset (rather than the number of samples).
 |  
 |  Args:
 |      workers: Number of workers to use in multithreading or
 |          multiprocessing.
 |      use_multiprocessing: Whether to use Python multiprocessing for
 |          parallelism. Setting this to `True` means that your
 |          dataset will be replicated in multiple forked processes.

In [8]:
import math

class MnistDataset(PyDataset):
        def __init__(self, x_set, y_set, batch_size, shuffle=True,seed=42,**kwargs):
            super().__init__(**kwargs)
            self.x, self.y = x_set, y_set
            self.batch_size = batch_size
            self.shuffle = shuffle
            self._rng = np.random.default_rng(seed)
            self.indices = np.arange(self.x.shape[0])
            if self.shuffle:
                self._rng.shuffle(self.indices)
            
        def __len__(self):
            # Return number of batches.
            return math.ceil(len(self.x) / self.batch_size)

        def __getitem__(self, idx):
            # Return x, y for batch idx.
            low = idx * self.batch_size
            # Cap upper bound at array length; the last batch may be smaller
            # if the total number of items is not a multiple of batch size.
            high = min(low + self.batch_size, len(self.x))
            batch_x = self.x[low:high]
            batch_y = self.y[low:high]

            return torch.flatten(self.x[self.indices[low:high]]/255, start_dim=1), self.y[self.indices[low:high]]

        def on_epoch_end(self) -> None:
            if self.shuffle:
                self._rng.shuffle(self.indices)
            

In [330]:
from torch.nn import functional as F

class MnistSAE(compresso.TopKSAE):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def encode(self, x):
        if isinstance(x, tuple):
            x,y = x
        if x.device != self.encoder.weight.device:
            x = x.to(self.encoder.weight.device)
        return self.sparsify(self.encoder(x))
                     
    def forward(self, x: Tensor):
        """
        Returns
        -------
        reconstruction : Tensor  ``(B, input_dim)``
        codes : Tensor  ``(B, hidden_dim)``   — sparse, exactly *k* nonzeros per row
        stats : dict[str, Tensor]
        """
        if isinstance(x, tuple):
            x,y = x
        
        if x.device != self.encoder.weight.device:
            x = x.to(self.encoder.weight.device)
            
        h = self.encoder(x)
        if self.pre_act is not None:
            h = self.pre_act(h)
        codes = self.sparsify(h)
        #codes = torch.nn.functional.normalize(codes, dim=-1)*self.k
        if self.tied:
            # encoder.weight is (hidden_dim, input_dim)
            # We want codes @ encoder.weight = (B, H) @ (H, D) = (B, D)
            # F.linear(input, weight, bias) computes input @ weight.T + bias
            # So pass encoder.weight.t() → (D, H), then F.linear does codes @ (D,H).T = codes @ (H,D)
            reconstruction = F.linear(codes, self.encoder.weight.t(), self.decoder_bias)
        else:
            reconstruction = self.decoder(codes)
        #reconstruction = torch.nn.functional.sigmoid(reconstruction)
        #reconstruction = torch.nn.functional.hardtanh(reconstruction, min_val=0, max_val=1)
        #reconstruction = torch.clip(reconstruction, min=0., max=1.)
        stats = self._compute_stats(x, reconstruction, codes)
        stats["BCE"]=torch.nn.functional.binary_cross_entropy(reconstruction, x, reduction="mean")
        return reconstruction, codes, stats

In [331]:
loader = MnistDataset(mnist.data,mnist.targets, 32)

In [332]:
model = MnistSAE(input_dim=28*28, hidden_dim=32, k=4, tied=False, pre_act=torch.nn.ReLU())
model.cuda()

MnistSAE(
  (encoder): Linear(in_features=784, out_features=32, bias=True)
  (sparsify): TopKSparsify(k=4, dim=-1, mode='values')
  (pre_act): ReLU()
  (decoder): Linear(in_features=32, out_features=784, bias=True)
)

In [333]:
model.sparsify

TopKSparsify(k=4, dim=-1, mode='values')

In [334]:
model.train()

MnistSAE(
  (encoder): Linear(in_features=784, out_features=32, bias=True)
  (sparsify): TopKSparsify(k=4, dim=-1, mode='values')
  (pre_act): ReLU()
  (decoder): Linear(in_features=32, out_features=784, bias=True)
)

In [335]:
EPOCHS=50

optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

# ── Training loop ───────────────────────────────────────────────────
n_batches = len(loader)
for epoch in range(1, EPOCHS + 1):
    epoch_loss = 0.0
    for i in range(n_batches):
        optimizer.zero_grad()
        batch = loader[i]
        recon, codes, stats = model(batch)
        loss = stats["BCE"]
    #    loss = stats["reconstruction_mse"]
        #loss += 1 - stats["cosine_similarity"]

        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    epoch_loss /= n_batches
    loader.on_epoch_end()
    if epoch % 1 == 0 or epoch == 1:
        print(
            f"Epoch {epoch:3d}/{EPOCHS}  "
            f"MSE={epoch_loss:.6f}  "
            f"cos_sim={stats['cosine_similarity'].item():.4f}  "
            f"active={stats['active_count'].item():.1f}  "
            f"dead={stats['dead_features'].item():.0f}/{model.hidden_dim}"
        )

# ── Final evaluation ────────────────────────────────────────────────

print("\n── Final stats ──")
print(f"  Reconstruction MSE : {stats['reconstruction_mse'].item():.6f}")
print(f"  Cosine similarity  : {stats['cosine_similarity'].item():.4f}")
print(f"  Active features    : {stats['active_count'].item():.1f} / {model.k}")
print(f"  Dead features      : {stats['dead_features'].item():.0f} / {model.hidden_dim}")
print(f"  Feature freq range : [{stats['activation_freq'].min().item():.4f}, "
      f"{stats['activation_freq'].max().item():.4f}]")


/pytorch/aten/src/ATen/native/cuda/Loss.cu:90: operator(): block: [0,0,0], thread: [9,0,0] Assertion `input_val >= zero && input_val <= one` failed.
/pytorch/aten/src/ATen/native/cuda/Loss.cu:90: operator(): block: [0,0,0], thread: [14,0,0] Assertion `input_val >= zero && input_val <= one` failed.
/pytorch/aten/src/ATen/native/cuda/Loss.cu:90: operator(): block: [0,0,0], thread: [16,0,0] Assertion `input_val >= zero && input_val <= one` failed.
/pytorch/aten/src/ATen/native/cuda/Loss.cu:90: operator(): block: [0,0,0], thread: [17,0,0] Assertion `input_val >= zero && input_val <= one` failed.
/pytorch/aten/src/ATen/native/cuda/Loss.cu:90: operator(): block: [0,0,0], thread: [23,0,0] Assertion `input_val >= zero && input_val <= one` failed.
/pytorch/aten/src/ATen/native/cuda/Loss.cu:90: operator(): block: [0,0,0], thread: [26,0,0] Assertion `input_val >= zero && input_val <= one` failed.
/pytorch/aten/src/ATen/native/cuda/Loss.cu:90: operator(): block: [18,0,0], thread: [98,0,0] Assertio

AcceleratorError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
st=loader[0][0].flatten()

In [134]:
bz=(
len(st[st>0.9]),
len(st[st>0.8]),
len(st[st>0.7]),
len(st[st>0.6]),
len(st[st>0.5]),
len(st[st>0.4]),
len(st[st>0.3]),
len(st[st>0.2]),
len(st[st>0.1]),
len(st)
)
[(round(1-(i+1)/10,1),bz[i]-((bz[1]-1) if i>0 else 0)) for i in range(len(bz))]


[(0.9, 2428),
 (0.8, 1),
 (0.7, 280),
 (0.6, 517),
 (0.5, 816),
 (0.4, 1064),
 (0.3, 1354),
 (0.2, 1600),
 (0.1, 1969),
 (0.0, 22314)]

In [325]:
st=model(loader[0][0])[0].flatten()
bz=(
len(st[st>0.9]),
len(st[st>0.8]),
len(st[st>0.7]),
len(st[st>0.6]),
len(st[st>0.5]),
len(st[st>0.4]),
len(st[st>0.3]),
len(st[st>0.2]),
len(st[st>0.1]),
len(st)
)
[(round(1-(i+1)/10,1),bz[i]-((bz[1]-1) if i>0 else 0)) for i in range(len(bz))]


[(0.9, 1622),
 (0.8, 1),
 (0.7, 168),
 (0.6, 331),
 (0.5, 492),
 (0.4, 693),
 (0.3, 865),
 (0.2, 1042),
 (0.1, 1229),
 (0.0, 23292)]

In [57]:
def encode(self, x):
        if isinstance(x, tuple):
            x,y = x
        if x.device != self.encoder.weight.device:
            x = x.to(self.encoder.weight.device)
        return self.sparsify(self.encoder(x))

In [216]:
j=7
pic, label = mnist[j]
tens = torch.from_numpy(np.array(pic))/255
print(encode(model,torch.flatten(tens[None, :, :])))
pic

tensor([    0.0000,     0.0000,     0.0000, -1167.7203,     0.0000,     0.0000,
            0.0000,     0.0000,     0.0000,     0.0000], device='cuda:0',
       grad_fn=<AddBackward0>)


In [210]:
j=12
pic, label = mnist[j]
tens = torch.from_numpy(np.array(pic))/255
print(encode(model,torch.flatten(tens[None, :, :])))
pic

tensor([   0.0000, 1266.3654,    0.0000,    0.0000,    0.0000,    0.0000,
           0.0000,    0.0000,    0.0000,    0.0000], device='cuda:0',
       grad_fn=<AddBackward0>)


In [219]:
dt=mnist.targets[:100]
[i  for i in range(len(dt)) if dt[i]==5]

[0, 11, 35, 47, 65]

In [220]:
j=0
pic, label = mnist[j]
tens = torch.from_numpy(np.array(pic))/255
print(encode(model,torch.flatten(tens[None, :, :])))
pic

tensor([772.4169,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
          0.0000,   0.0000,   0.0000], device='cuda:0', grad_fn=<AddBackward0>)


In [221]:
j=11
pic, label = mnist[j]
tens = torch.from_numpy(np.array(pic))/255
print(encode(model,torch.flatten(tens[None, :, :])))
pic

tensor([  0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000, 721.6852,
          0.0000,   0.0000,   0.0000], device='cuda:0', grad_fn=<AddBackward0>)


In [222]:
j=35
pic, label = mnist[j]
tens = torch.from_numpy(np.array(pic))/255
print(encode(model,torch.flatten(tens[None, :, :])))
pic

tensor([   0.0000,    0.0000,    0.0000,    0.0000,    0.0000,    0.0000,
           0.0000, -561.2817,    0.0000,    0.0000], device='cuda:0',
       grad_fn=<AddBackward0>)


In [79]:
j=47
pic, label = mnist[j]
tens = torch.from_numpy(np.array(pic))/255
print(encode(model,torch.flatten(tens[None, :, :])))
pic

tensor([   0.0000,    0.0000,    0.0000, -118.9888,    0.0000,    0.0000,
           0.0000,    0.0000,    0.0000,    0.0000,    0.0000,  115.3218,
           0.0000,    0.0000,    0.0000,    0.0000], device='cuda:0',
       grad_fn=<AddBackward0>)


In [326]:
j=65
pic, label = mnist[j]
tens = torch.from_numpy(np.array(pic))/255
rep=encode(model,torch.flatten(tens[None, :, :]))
print(rep)
pic

tensor([ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000, 42.3598,  0.0000,
         0.0000,  0.0000,  0.0000, 40.6754,  0.0000,  0.0000,  0.0000,  0.0000,
        41.4839,  0.0000,  0.0000, 32.8949,  0.0000,  0.0000,  0.0000,  0.0000],
       device='cuda:0', grad_fn=<AddBackward0>)


In [327]:
import PIL

PIL.Image.fromarray((torch.nn.functional.relu(model.decoder(rep)).cpu().detach().numpy()*255).astype("uint8").reshape([28,28]))

In [328]:
j=1
pic, label = mnist[j]
tens = torch.from_numpy(np.array(pic))/255
rep=encode(model,torch.flatten(tens[None, :, :]))
print(rep)
pic

tensor([ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000, 86.1352,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000, 86.1974,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
        85.7541,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000, 76.9938],
       device='cuda:0', grad_fn=<AddBackward0>)


In [329]:
import PIL

PIL.Image.fromarray((torch.nn.functional.relu(model.decoder(rep)).cpu().detach().numpy()*255).astype("uint8").reshape([28,28]))

In [37]:
tens

tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000

In [ ]:
from __future__ import annotations

from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

from compresso.nn.sparsify import TopKSparsify

class TopKSAE(nn.Module):
    """Sparse autoencoder with hard top-k bottleneck.

    Parameters
    ----------
    input_dim : int
        Dimensionality of the input (and reconstruction).
    hidden_dim : int
        Width of the sparse code layer.
    k : int
        Number of active features per sample.
    tied : bool
        If ``True``, decoder weight is the transpose of the encoder weight.
    pre_act : nn.Module | None
        Optional activation applied to encoder output *before* sparsification.
    """

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        k: int,
        tied: bool = False,
        pre_act: Optional[nn.Module] = None,
    ) -> None:
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.k = k
        self.tied = tied

        self.encoder = nn.Linear(input_dim, hidden_dim)
        self.sparsify = TopKSparsify(k=k, dim=-1, mode="values")

        if pre_act is not None:
            self.pre_act = pre_act
        else:
            self.pre_act = None

        if tied:
            # Only a bias for the decoder path; weight is encoder.weight.T
            self.decoder_bias = nn.Parameter(torch.zeros(input_dim))
        else:
            self.decoder = nn.Linear(hidden_dim, input_dim)

    # ------------------------------------------------------------------
    # Public helpers
    # ------------------------------------------------------------------

    def get_decoder_weight(self) -> Tensor:
        """Return the effective decoder weight matrix ``(input_dim, hidden_dim)``."""
        if self.tied:
            return self.encoder.weight.t()
        return self.decoder.weight

    # ------------------------------------------------------------------
    # Forward
    # ------------------------------------------------------------------

    def forward(self, x: Tensor):
        """
        Returns
        -------
        reconstruction : Tensor  ``(B, input_dim)``
        codes : Tensor  ``(B, hidden_dim)``   — sparse, exactly *k* nonzeros per row
        stats : dict[str, Tensor]
        """
        h = self.encoder(x)
        if self.pre_act is not None:
            h = self.pre_act(h)
        codes = self.sparsify(h)

        if self.tied:
            # encoder.weight is (hidden_dim, input_dim)
            # We want codes @ encoder.weight = (B, H) @ (H, D) = (B, D)
            # F.linear(input, weight, bias) computes input @ weight.T + bias
            # So pass encoder.weight.t() → (D, H), then F.linear does codes @ (D,H).T = codes @ (H,D)
            reconstruction = F.linear(codes, self.encoder.weight.t(), self.decoder_bias)
        else:
            reconstruction = self.decoder(codes)

        stats = self._compute_stats(x, reconstruction, codes)
        return reconstruction, codes, stats

    # ------------------------------------------------------------------
    # Stats
    # ------------------------------------------------------------------

    def _compute_stats(self, x: Tensor, recon: Tensor, codes: Tensor) -> dict:
        active_mask = codes != 0  # (B, H)

        active_count = active_mask.float().sum(dim=-1).mean()
        activation_freq = active_mask.float().mean(dim=0)  # (H,)
        dead_features = (activation_freq == 0).sum()

        diff = x - recon
        reconstruction_mse = (diff * diff).mean()

        # Cosine similarity (per sample, then average)
        cos = F.cosine_similarity(x, recon, dim=-1).mean()

        return {
            "active_count": active_count,
            "activation_freq": activation_freq,
            "reconstruction_mse": reconstruction_mse,
            "cosine_similarity": cos,
            "dead_features": dead_features,
        }


In [63]:
loader = MnistDataset(mnist.data,mnist.targets, 128)

(tensor([[[0, 0, 0,  ..., 0, 0, 0],
          [0, 0, 0,  ..., 0, 0, 0],
          [0, 0, 0,  ..., 0, 0, 0],
          ...,
          [0, 0, 0,  ..., 0, 0, 0],
          [0, 0, 0,  ..., 0, 0, 0],
          [0, 0, 0,  ..., 0, 0, 0]],
 
         [[0, 0, 0,  ..., 0, 0, 0],
          [0, 0, 0,  ..., 0, 0, 0],
          [0, 0, 0,  ..., 0, 0, 0],
          ...,
          [0, 0, 0,  ..., 0, 0, 0],
          [0, 0, 0,  ..., 0, 0, 0],
          [0, 0, 0,  ..., 0, 0, 0]],
 
         [[0, 0, 0,  ..., 0, 0, 0],
          [0, 0, 0,  ..., 0, 0, 0],
          [0, 0, 0,  ..., 0, 0, 0],
          ...,
          [0, 0, 0,  ..., 0, 0, 0],
          [0, 0, 0,  ..., 0, 0, 0],
          [0, 0, 0,  ..., 0, 0, 0]],
 
         ...,
 
         [[0, 0, 0,  ..., 0, 0, 0],
          [0, 0, 0,  ..., 0, 0, 0],
          [0, 0, 0,  ..., 0, 0, 0],
          ...,
          [0, 0, 0,  ..., 0, 0, 0],
          [0, 0, 0,  ..., 0, 0, 0],
          [0, 0, 0,  ..., 0, 0, 0]],
 
         [[0, 0, 0,  ..., 0, 0, 0],
          [0